In [257]:
import os

os.environ["PYSPARK_PYTHON"] = "python"
os.environ["PYSPARK_DRIVER_PYTHON"] = "python"

# STEP 1: SETUP PYSPARK JOB

## Basic Spark Setup

In [258]:
from pyspark.sql import SparkSession

def create_spark_session():
    return SparkSession.builder \
        .appName("SalaryDetectionJob") \
        .config("spark.python.worker.timeout", "120") \
        .config("spark.executor.heartbeatInterval", "60s") \
        .getOrCreate()

## Entry point

In [259]:
spark = create_spark_session()

df = spark.read.csv(
        "C:/Users/ACER/Downloads/salary-detector/data/test_scenarios.csv",
        header=True,
        inferSchema=True
    )

df.show(5)

+--------------------+----------+-------------------+------+------+-------+--------+
|          CustomerId|      Date|Transaction Details|  Type|Amount|Balance|Category|
+--------------------+----------+-------------------+------+------+-------+--------+
|# C1 → Normal mon...|      NULL|               NULL|  NULL|  NULL|   NULL|    NULL|
|                  C1|2024-09-05|NEFT/TCS SALARY SEP|Credit| 60000|  70000|  Salary|
|                  C1|2024-10-05|NEFT/TCS SALARY OCT|Credit| 60500| 130000|  Salary|
|                  C1|2024-11-05|NEFT/TCS SALARY NOV|Credit| 60000| 190000|  Salary|
|                  C1|2024-11-07|         UPI/Swiggy| Debit|   500| 189500|    Food|
+--------------------+----------+-------------------+------+------+-------+--------+
only showing top 5 rows


# STEP 2: CLEAN & STANDARDIZE DATA

In [260]:
from pyspark.sql.functions import col, to_date

df = df.withColumn("Date", to_date(col("Date"), "yyyy-MM-dd"))

df = df.withColumn("Amount", col("Amount").cast("double"))
df = df.withColumn("Balance", col("Balance").cast("double"))

df = df.filter(col("Date").isNotNull())

# STEP 3: FILTER CREDIT TRANSACTIONS

## PySpark version:

In [261]:
from pyspark.sql.functions import upper

credit_df = df.filter(
    (upper(col("Type")) == "CREDIT") &
    (col("Amount") >= 3000)
)

# STEP 4: SENDER EXTRACTION

In [262]:
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType

def extract_sender(details):
    if not details:
        return "UNKNOWN"

    details = details.upper()
    parts = details.split("/")

    if len(parts) >= 2:
        sender = parts[1]
        words = sender.split()

        sender = " ".join(words[:2])

        if sender.replace(" ", "").isdigit():
            return f"ANON_{sender.replace(' ', '')}"

        return sender.strip()

    return "UNKNOWN"

extract_sender_udf = udf(extract_sender, StringType())

In [263]:
credit_df = credit_df.withColumn("sender", col("Transaction Details"))

# STEP 5: MERGE SAME-DAY TRANSACTIONS

In [264]:
from pyspark.sql.functions import sum as spark_sum

daily_df = credit_df.groupBy(
    "CustomerId", "sender", "Date"
).agg(
    spark_sum("Amount").alias("daily_amount")
)

In [265]:
from pyspark.sql.functions import collect_list

# STEP 6: COMPUTE INTERVALS

In [266]:
from pyspark.sql.window import Window
from pyspark.sql.functions import lag, datediff

window_spec = Window.partitionBy("CustomerId", "sender").orderBy("Date")

daily_df = daily_df.withColumn(
    "prev_date",
    lag("Date").over(window_spec)
)

daily_df = daily_df.withColumn(
    "interval_days",
    datediff(col("Date"), col("prev_date"))
)

# STEP 7: AGGREGATE FEATURES

## Feature aggregation:

In [267]:
from pyspark.sql.functions import collect_list

features_df = daily_df.groupBy(
    "CustomerId", "sender"
).agg(
    avg("interval_days").alias("interval_mean"),
    stddev("interval_days").alias("interval_std"),
    avg("daily_amount").alias("amount_mean"),
    stddev("daily_amount").alias("amount_std"),
    count("*").alias("count"),
    collect_list("daily_amount").alias("salary_history")  # ✅ THIS
)

In [268]:
from pyspark.sql.functions import col, lower

# 1. periodic check
features_df = features_df.withColumn(
    "is_periodic",
    (
        col("interval_mean").isNotNull() &
        (
            col("interval_mean").between(24, 35) |   # monthly
            col("interval_mean").between(5, 10)  |   # weekly
            col("interval_mean").between(10, 22)     # semi-monthly / flexible
        )
    )
)

# 2. stable amount
features_df = features_df.withColumn(
    "is_stable_amount",
    col("amount_std") < 10000
)

# 3. repetition
features_df = features_df.withColumn(
    "has_repetition",
    col("count") >= 2
)

features_df = features_df.withColumn(
    "has_salary_keyword",
    lower(col("sender")).rlike("salary|sal|payroll")
)

In [269]:
from pyspark.sql.functions import when

features_df = features_df.withColumn(
    "salary_confidence",
    when(col("count") >= 3, "HIGH")
    .when(col("count") == 2, "MEDIUM")
    .otherwise("LOW")
)

In [270]:
salary_candidates = features_df.filter(
    col("is_periodic") &
    col("is_stable_amount") &
    col("has_repetition") &
    (
        col("has_salary_keyword") | 
        (col("amount_mean") > 20000)   # fallback for anonymous salary
    )
)

In [271]:
from pyspark.sql.functions import sort_array

features_df = features_df.withColumn(
    "salary_history",
    sort_array(col("salary_history"))
)

In [272]:
from pyspark.sql.functions import coalesce, lit

features_df = features_df.withColumn(
    "interval_std",
    coalesce(col("interval_std"), lit(0.0))
)

features_df = features_df.withColumn(
    "amount_std",
    coalesce(col("amount_std"), lit(0.0))
)

features_df = features_df.withColumn(
    "amount_mean",
    coalesce(col("amount_mean"), lit(1.0))  # avoid division issues
)

# STEP 8: APPLY SALARY RULES

In [273]:
features_df = features_df.withColumn(
    "is_periodic",
    (
        col("interval_mean").isNotNull() &
        (
            col("interval_mean").between(24, 35) |   # monthly
            col("interval_mean").between(5, 10)  |   # weekly
            col("interval_mean").between(10, 22)     # semi-monthly / flexible
        )
    )
)

features_df = features_df.withColumn(
    "is_stable_amount",
    col("amount_std") < 10000
)

features_df = features_df.withColumn(
    "has_repetition",
    col("count") >= 2
)

salary_candidates = features_df.filter(
    col("is_periodic") &
    col("is_stable_amount") &
    col("has_repetition") &
    (col("count") >= 3) 
)

# STEP 9: SCORING

## Score calculation:

In [274]:
from pyspark.sql.functions import lit

salary_candidates = salary_candidates.withColumn(
    "time_score",
    1 / (1 + col("interval_std"))
)

salary_candidates = salary_candidates.withColumn(
    "amount_score",
    1 / (1 + (col("amount_std") / (col("amount_mean") + 1)))
)

salary_candidates = salary_candidates.withColumn(
    "salary_boost",
    (col("amount_mean") > 30000).cast("int")
)

salary_candidates = salary_candidates.withColumn(
    "count_score",
    (col("count") / 5)
)

salary_candidates = salary_candidates.withColumn(
    "final_score",
    (
        0.5 * col("time_score") +
        0.25 * col("amount_score") +
        0.15 * col("count_score") +
        0.1 * col("salary_boost")
    )
)

In [275]:
from pyspark.sql.functions import coalesce, lit

features_df = features_df.withColumn(
    "interval_mean",
    coalesce(col("interval_mean"), lit(999))
)

# STEP 10: PICK BEST SENDER PER CUSTOMER

## Window ranking

In [276]:
from pyspark.sql.functions import row_number

rank_window = Window.partitionBy("CustomerId").orderBy(col("final_score").desc())

ranked_df = salary_candidates.withColumn(
    "rank",
    row_number().over(rank_window)
)

best_df = ranked_df.filter(col("rank") == 1)

In [277]:
rank_window = Window.partitionBy("CustomerId").orderBy(col("final_score").desc())

ranked_df = salary_candidates.withColumn(
    "rank",
    row_number().over(rank_window)
)

best_df = ranked_df.filter(col("rank") == 1)

In [278]:
best_df.show()

+----------+------------------+-------------+------------------+------------------+------------------+-----+--------------------+-----------+----------------+--------------+------------------+-----------------+-------------------+------------------+------------+-----------+-------------------+----+
|CustomerId|            sender|interval_mean|      interval_std|       amount_mean|        amount_std|count|      salary_history|is_periodic|is_stable_amount|has_repetition|has_salary_keyword|salary_confidence|         time_score|      amount_score|salary_boost|count_score|        final_score|rank|
+----------+------------------+-------------+------------------+------------------+------------------+-----+--------------------+-----------+----------------+--------------+------------------+-----------------+-------------------+------------------+------------+-----------+-------------------+----+
|        C5|UPI/Client Payment|         18.5|2.1213203435596424|21666.666666666668| 7637.62615825973

## STEP 11: OUTPUT

In [279]:
features_df.select(
    "CustomerId",
    "count",
    "interval_mean",
    "is_periodic",
    "is_stable_amount",
    "has_repetition"
).show(truncate=False)

+----------+-----+-------------+-----------+----------------+--------------+
|CustomerId|count|interval_mean|is_periodic|is_stable_amount|has_repetition|
+----------+-----+-------------+-----------+----------------+--------------+
|C1        |1    |999.0        |false      |true            |false         |
|C1        |1    |999.0        |false      |true            |false         |
|C1        |1    |999.0        |false      |true            |false         |
|C2        |2    |30.0         |true       |true            |true          |
|C3        |2    |30.0         |true       |true            |true          |
|C5        |3    |18.5         |true       |true            |true          |
|C6        |3    |7.0          |true       |true            |true          |
|C7        |2    |30.0         |true       |true            |true          |
|C7        |2    |30.0         |true       |true            |true          |
|C8        |3    |30.5         |true       |true            |true          |